# ACS Data Analysis

* Add description of what the purpose of this notebook is
* Overview of the steps etc
* 

### Setup

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import camelot
import calendar
import re
import matplotlib.pyplot as plt

## 1. ACSDP05 (demographic data) (ACS data 1/3)

In [2]:
ACS5 = pd.read_csv('../data/ACSDP05.csv')

ACS5.shape

(99, 237)

In [3]:
ACS5.head(30)

,Label (Grouping),"Alameda County, California!!Estimate","Alameda County, California!!Margin of Error","Alameda County, California!!Percent","Alameda County, California!!Percent Margin of Error","Alpine County, California!!Estimate","Alpine County, California!!Margin of Error","Alpine County, California!!Percent","Alpine County, California!!Percent Margin of Error","Amador County, California!!Estimate",...,"Yolo County, California!!Percent","Yolo County, California!!Percent Margin of Error","Yuba County, California!!Estimate","Yuba County, California!!Margin of Error","Yuba County, California!!Percent","Yuba County, California!!Percent Margin of Error",ZCTA5 99641!!Estimate,ZCTA5 99641!!Margin of Error,ZCTA5 99641!!Percent,ZCTA5 99641!!Percent Margin of Error
0,SEX AND AGE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Total population,"1,651,949",*****,"1,651,949",(X),"1,695",±234,"1,695",(X),"41,029",...,"217,782",(X),"83,079",*****,"83,079",(X),766.0,±166,766,(X)
2,Male,"819,520",*****,49.6%,*****,885,±148,52.2%,±4.8,"22,409",...,48.7%,±0.1,"42,183",±211,50.8%,±0.3,356.0,±77,46.5%,±5.5
3,Female,"832,429",*****,50.4%,*****,810,±139,47.8%,±4.8,"18,620",...,51.3%,±0.1,"40,896",±211,49.2%,±0.3,410.0,±108,53.5%,±5.5
4,Sex ratio (males per 100 females),98.4,*****,(X),(X),109.3,±21.4,(X),(X),120.3,...,(X),(X),103.1,±1.0,(X),(X),86.8,±20.0,(X),(X)
5,Under 5 years,"88,228",*****,5.3%,*****,161,±85,9.5%,±4.6,"1,587",...,5.0%,±0.1,"6,099",±7,7.3%,±0.1,103.0,±39,13.4%,±3.6
6,5 to 9 years,"91,717","±1,746",5.6%,±0.1,83,±39,4.9%,±2.2,"1,712",...,5.7%,±0.3,"7,148",±612,8.6%,±0.7,112.0,±36,14.6%,±2.8
7,10 to 14 years,"94,521","±1,746",5.7%,±0.1,56,±30,3.3%,±1.8,"1,857",...,6.0%,±0.3,"6,015",±606,7.2%,±0.7,74.0,±34,9.7%,±3.5
8,15 to 19 years,"96,042",±147,5.8%,±0.1,115,±58,6.8%,±3.0,"1,776",...,9.7%,±0.1,"5,568",±148,6.7%,±0.2,58.0,±19,7.6%,±2.2
9,20 to 24 years,"91,539",±148,5.5%,±0.1,102,±73,6.0%,±4.2,"1,593",...,13.8%,±0.1,"5,311",±183,6.4%,±0.2,68.0,±26,8.9%,±3.1


* Selecting just the percent and estimate columns

In [4]:
estimate_cols = [c for c in ACS5.columns if c.endswith('!!Estimate')]
percent_cols = [c for c in ACS5.columns if c.endswith('!!Percent')]

* Cleaning the county names

In [5]:
county_names = [c.split('!!')[0].strip() for c in percent_cols]

* Searching/selecting for median age row 

In [6]:
median_idx = ACS5['Label (Grouping)'].astype(str).str.contains('Median age', na=False)

* Constructing final dataset by locating desired measures and renaming columns

In [7]:
ACS5_final = pd.DataFrame({
    'county': county_names,
    'median_age': ACS5.loc[median_idx, estimate_cols].iloc[0].values,
    'pct_white': ACS5.loc[71, percent_cols].values,
    'pct_black': ACS5.loc[72, percent_cols].values,
    'pct_native': ACS5.loc[73, percent_cols].values,
    'pct_asian': ACS5.loc[74, percent_cols].values,
    'pct_other': ACS5.loc[76, percent_cols].values,
    'pct_hispanic': ACS5.loc[79, percent_cols].values,
})

In [8]:
ACS5_final

,county,median_age,pct_white,pct_black,pct_native,pct_asian,pct_other,pct_hispanic
0,"Alameda County, California",38.7,41.8%,12.1%,2.1%,36.1%,19.1%,23.3%
1,"Alpine County, California",41.1,71.0%,0.0%,28.1%,0.5%,8.3%,14.7%
2,"Amador County, California",49.9,89.0%,3.7%,5.0%,3.6%,12.3%,15.5%
3,"Butte County, California",36.3,84.3%,3.1%,4.3%,7.1%,13.6%,19.5%
4,"Calaveras County, California",52.0,91.6%,1.6%,3.8%,3.2%,10.8%,13.9%
5,"Colusa County, California",35.7,70.6%,2.1%,3.2%,2.1%,42.1%,62.3%
6,"Contra Costa County, California",40.3,56.4%,10.8%,2.6%,22.4%,22.6%,27.3%
7,"Del Norte County, California",40.8,78.1%,3.9%,10.3%,4.8%,14.0%,19.6%
8,"El Dorado County, California",46.1,88.8%,1.7%,2.5%,7.5%,8.9%,14.2%
9,"Fresno County, California",33.2,61.6%,6.0%,3.1%,12.8%,38.1%,54.1%


#### Convert to numeric

In [17]:
ACS5_final['county'] = ACS5_final['county'].str.replace(', California','')
ACS5_final['median_age'] = ACS5_final['median_age'].astype(float)

In [10]:
def pct_to_numeric(pct_str):
    return float(pct_str.replace('%',''))

In [13]:
pct_cols = [cname for cname in ACS5_final.columns if cname.startswith('pct_')]

In [15]:
ACS5_final[pct_cols] = ACS5_final[pct_cols].map(pct_to_numeric)

In [18]:
ACS5_final

,county,median_age,pct_white,pct_black,pct_native,pct_asian,pct_other,pct_hispanic
0,Alameda County,38.7,41.8,12.1,2.1,36.1,19.1,23.3
1,Alpine County,41.1,71.0,0.0,28.1,0.5,8.3,14.7
2,Amador County,49.9,89.0,3.7,5.0,3.6,12.3,15.5
3,Butte County,36.3,84.3,3.1,4.3,7.1,13.6,19.5
4,Calaveras County,52.0,91.6,1.6,3.8,3.2,10.8,13.9
5,Colusa County,35.7,70.6,2.1,3.2,2.1,42.1,62.3
6,Contra Costa County,40.3,56.4,10.8,2.6,22.4,22.6,27.3
7,Del Norte County,40.8,78.1,3.9,10.3,4.8,14.0,19.6
8,El Dorado County,46.1,88.8,1.7,2.5,7.5,8.9,14.2
9,Fresno County,33.2,61.6,6.0,3.1,12.8,38.1,54.1


* Save cleaned and converted version back to `data` folder

In [19]:
ACS5_final.to_csv('../data/ACSDP05_final.csv', index=False)

## 2. ACSDP03 (socioeconomic data) (ACS data 3/3)

In [25]:
ACS3 = pd.read_csv('../data/ACSDP03.csv')

Selecting just the percent and estimate columns

In [26]:
estimate_cols = [c for c in ACS3.columns if c.endswith('!!Estimate')]
percent_cols = [c for c in ACS3.columns if c.endswith('!!Percent')]

Cleaning the county names

In [27]:
county_names = [c.split('!!')[0].strip() for c in percent_cols]

Constructing final dataset by locating desired measures and renaming columns

In [28]:
ACS3_final = pd.DataFrame({
    'county': county_names,

    'median_household_income': ACS3.loc[67, estimate_cols].values,
    'unemployment_rate': ACS3.loc[9, percent_cols].values,

    'pct_agriculture_mining': ACS3.loc[36, percent_cols].values,
    'pct_construction': ACS3.loc[37, percent_cols].values,
    'pct_manufacturing': ACS3.loc[38, percent_cols].values,
    'pct_wholesale_trade': ACS3.loc[39, percent_cols].values,
    'pct_retail_trade': ACS3.loc[40, percent_cols].values,
    'pct_transportation_warehousing_utilities': ACS3.loc[41, percent_cols].values,
    'pct_information': ACS3.loc[42, percent_cols].values,
    'pct_real_estate_rental_leasing': ACS3.loc[43, percent_cols].values,
    'pct_admin_waste_services': ACS3.loc[44, percent_cols].values,
    'pct_health_care_social_assistance': ACS3.loc[45, percent_cols].values,
    'pct_accommodation_food_services': ACS3.loc[46, percent_cols].values,
    'pct_other_services': ACS3.loc[47, percent_cols].values,
    'pct_public_administration': ACS3.loc[48, percent_cols].values,
})

In [32]:
ACS3_final

,county,median_household_income,unemployment_rate,pct_agriculture_mining,pct_construction,pct_manufacturing,pct_wholesale_trade,pct_retail_trade,pct_transportation_warehousing_utilities,pct_information,pct_real_estate_rental_leasing,pct_admin_waste_services,pct_health_care_social_assistance,pct_accommodation_food_services,pct_other_services,pct_public_administration
0,"Alameda County, California","122,488",4.9%,0.4%,5.3%,9.7%,2.1%,8.8%,5.5%,3.8%,6.0%,20.7%,21.8%,7.6%,4.5%,3.7%
1,"Alpine County, California","101,125",4.9%,9.0%,2.0%,5.7%,0.6%,19.7%,5.1%,0.8%,2.2%,10.1%,20.1%,21.2%,0.0%,3.6%
2,"Amador County, California","74,853",6.0%,4.4%,9.5%,5.3%,1.4%,10.1%,4.4%,1.4%,5.3%,10.8%,15.3%,12.6%,7.7%,11.6%
3,"Butte County, California","66,085",7.1%,3.7%,6.7%,6.4%,2.0%,12.0%,4.3%,2.1%,4.9%,10.7%,26.7%,11.8%,4.4%,4.4%
4,"Calaveras County, California","77,526",6.2%,3.4%,10.6%,5.5%,1.7%,11.2%,5.0%,1.7%,5.9%,11.9%,19.8%,8.6%,5.0%,9.7%
5,"Colusa County, California","69,619",7.4%,22.0%,4.7%,12.4%,2.2%,8.8%,7.8%,0.4%,1.7%,4.1%,15.8%,8.5%,4.6%,7.0%
6,"Contra Costa County, California","120,020",5.8%,0.6%,7.7%,6.2%,2.1%,10.1%,5.9%,2.5%,7.6%,17.2%,22.8%,8.2%,4.9%,4.3%
7,"Del Norte County, California","61,149",6.3%,5.2%,6.9%,4.6%,2.9%,12.0%,3.1%,0.8%,2.2%,6.6%,26.8%,11.2%,3.4%,14.4%
8,"El Dorado County, California","99,246",4.6%,1.6%,8.5%,6.2%,1.6%,9.6%,3.8%,1.7%,7.6%,14.0%,20.0%,13.1%,5.2%,7.1%
9,"Fresno County, California","67,756",8.6%,8.7%,6.3%,6.4%,3.2%,10.4%,6.4%,1.2%,4.3%,9.3%,24.9%,8.1%,4.7%,6.3%


## 3. ACSDP02 (educational attainment data) (ACS data 3/3)

In [33]:
ACS2 = pd.read_csv('../data/ACSDP02.csv')

Selecting just the percent column

In [12]:
percent_cols = [c for c in ACS2.columns if c.endswith('!!Percent')]

Cleaning the county names

In [13]:
county_names = [c.split('!!')[0].strip() for c in percent_cols]

Constructing final dataset by locating desired measure and renaming column

In [14]:
ACS2_final = pd.DataFrame({
    'county': county_names,
    'pct_bachelors_degree': ACS2.loc[75, percent_cols].values
})

Merging all three cleaned ACS datasets

In [15]:
final_df = (
    ACS5_final
    .merge(ACS3_final, on='county', how='left')
    .merge(ACS2_final, on='county', how='left')
)

Note: I encountered significant difficulty in my attempts to reshape the data (specifically, ensuring that the correct values were aligned with the right measures for each county after reshaping) so I relied heavily on ChatGPT to clean/reshape the data.